# LoginGuard AI — données réelles LANL
Étude des authentifications anonymisées. Les annotations red team sont partielles et ne constituent pas une vérité terrain exhaustive.

In [ ]:
from pathlib import Path
import sys, json, joblib
import pandas as pd
import plotly.express as px
root = Path.cwd() if Path('src').exists() else Path.cwd().parent
sys.path.insert(0, str(root))
from src import lanl

## Source et objectif
Détecter des comportements atypiques dans les trois premiers jours LANL. Source : https://csr.lanl.gov/data/cyber1/ ; DOI 10.17021/1179829. Import préalable : `python -m src.import_lanl`, puis `python -m src.lanl`. Aucune IP ou date civile n’est inventée.

In [ ]:
data = pd.read_csv(lanl.DATA/'login_logs.csv')
manifest = json.loads((lanl.DATA/'prepared_manifest.json').read_text(encoding='utf-8'))
print(manifest['rows'], manifest['users'])
data.head()

## Exploration et validation
Les événements possèdent un identifiant de ligne source. Les doublons de contenu peuvent être des événements distincts. Les champs obligatoires absents sont rejetés.

In [ ]:
print(data[lanl.RAW].isna().sum())
print('Identifiants dupliqués :', data.event_id.duplicated().sum())
lanl.validate(data)
data.groupby(['relative_day','outcome']).size()

## Historique causal et caractéristiques
La fenêtre comprend les événements dans [t−600,t[. Les événements simultanés ne se voient pas mutuellement. La nouveauté est relative au sous-ensemble observé. Seules les sept caractéristiques numériques prévues sont transmises au modèle.

In [ ]:
features = lanl.engineer(data[lanl.RAW])
assert not {'redteam_label','user_id','event_id'} & set(lanl.FEATURES)
lanl.matrix(features).describe()

## Découpage chronologique et entraînement
Les deux premiers jours entraînent une forêt de 200 arbres, contamination 1 %, seed 42. Aucune annotation n’entre dans l’apprentissage. Le troisième jour reste réservé à l’évaluation. Le modèle du notebook est sauvegardé séparément.

In [ ]:
train = data[data.time < lanl.CUTOFF]
test = data[data.time >= lanl.CUTOFF]
bundle = lanl.train(train, root/'artifacts/notebook_lanl.joblib')
print(len(train),len(test))

## Prédictions et scores
La forêt renvoie −1 pour une anomalie. Le tableau de bord traduit cela en 1. Le score est l’opposé de decision_function : un score positif dépasse le seuil. Ni le score ni le percentile ne sont une probabilité d’attaque.

In [ ]:
scored = lanl.predict(test,bundle)
scored[['event_id','anomaly_score','prediction']].head()

## Évaluation et comparaison aux règles
Les métriques sont relatives aux annotations partielles : les alertes non annotées ne sont pas nécessairement des faux positifs. Le rappel repose sur très peu d’événements.

In [ ]:
results=lanl.evaluate(test,bundle)
pd.DataFrame(results).T

In [ ]:
px.imshow(results['Isolation Forest']['confusion_matrix'], x=['Habituel','À vérifier'], y=['Non annoté','Red team'], text_auto=True).show()
px.histogram(scored,x='anomaly_score',color=scored.redteam_label.astype(str),barmode='overlay').show()

## Conclusion
Sur ce sous-ensemble, le modèle manque une grande partie des annotations red team ; une accuracy élevée ne démontre pas une bonne détection. Les règles basées sur des fréquences faibles déclenchent excessivement. Une étude plus longue avec validation temporelle indépendante est nécessaire. Les données restent réelles, anonymisées et limitées à la cohorte annoncée.